# Pet Breed Classifier

In this project, I am gonna build a pet breed classifier, using the Oxford-IIIT Pet Dataset: https://www.robots.ox.ac.uk/~vgg/data/pets/

The dataset has 37 different breed categories. 25 for dogs, 12 for cats. With a total of 7349 images.

In this notebook, I am gonna convert the useful code for training model, creating dataloaders, saving/loading the model, etc, into a series of Python scripts, so that I won't have to write it again and again. I am just gonna work on the useful code only for now.

In [5]:
from pathlib import Path
going_modular_dir = Path('going_modular')

if going_modular_dir.exists():
    print('Directory already exists')
else:
    going_modular_dir.mkdir(parents=True, exist_ok=True)
    print('Created going_modular directory')

Directory already exists


In [6]:
%%writefile going_modular/engine.py

import torch
from tqdm.auto import tqdm


def accuracy_fn(y_pred_label, y_true):
    correct = torch.eq(y_pred_label, y_true).sum().item()
    accuracy = correct/len(y_pred_label)
    return accuracy

def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               optimizer: torch.optim.Optimizer,
               loss_fn: torch.nn.Module,
               device: torch.device):

    model.train()
    train_loss, train_acc = 0, 0

    for X, y in dataloader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        y_pred = model(X)
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()
        loss.backward()
        optimizer.step()

        y_pred_class = torch.argmax(y_pred, dim=1)
        train_acc += accuracy_fn(y_pred_class, y)

    train_loss = train_loss / len(dataloader)
    train_acc = train_acc / len(dataloader)
    return train_loss, train_acc

def test_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              device: torch.device):

    model.eval()
    test_loss, test_acc = 0,0

    with torch.inference_mode():
        for X,y in dataloader:
            X, y = X.to(device), y.to(device)
            test_logits = model(X)
            loss = loss_fn(test_logits, y)
            test_loss += loss.item()

            test_pred_class = torch.argmax(test_logits, dim=1)
            test_acc += accuracy_fn(test_pred_class, y)

    test_loss = test_loss / len(dataloader)
    test_acc = test_acc / len(dataloader)
    return test_loss, test_acc

def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          epochs: int,
          device: torch.device):

    results = {'train_loss': [], 
               'train_acc': [], 
               'test_loss': [], 
               'test_acc': []}
    
    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(model=model,
                                           dataloader=train_dataloader,
                                           optimizer=optimizer,
                                           loss_fn=loss_fn,
                                           device=device)
        test_loss, test_acc = test_step(model=model,
                                        dataloader=test_dataloader,
                                        loss_fn=loss_fn,
                                        device=device)

        print(f'Epoch: {epoch+1} || Train Loss: {train_loss:.2f} || Train Accuracy: {(train_acc*100):.2f} || Test Loss: {test_loss:.2f} || Test Accuracy: {(test_acc*100):.2f}')

        results['train_loss'].append(train_loss)
        results['train_acc'].append(train_acc)
        results['test_loss'].append(test_loss)
        results['test_acc'].append(test_acc)

    return results

Overwriting going_modular/engine.py


In [7]:
%%writefile going_modular/utils.py

import torch
from pathlib import Path

def save_model(model: torch.nn.Module,
               model_name: str,
               target_dir: str):

    target_dir_path = Path(target_dir)
    target_dir_path.mkdir(parents=True, exist_ok=True)

    assert model_name.endswith(('.pth', '.pt')), 'model_name should end with "pth" or "pt"'
    model_save_path = target_dir_path / model_name

    print('Saving the model..')
    torch.save(obj=model.state_dict(), f=model_save_path)
    print('Model has been saved successfully.')

def load_model(model: torch.nn.Module,
               model_path: str,
               device:str = 'cpu'):  
    state_dict = torch.load(f=model_path, map_location=device)
    model.load_state_dict(state_dict=state_dict)
    model.to(device)
    model.eval()
    return model

Overwriting going_modular/utils.py


In [8]:
%%writefile going_modular/data_setup.py

import torch

NUM_WORKERS = 2
def create_dataloaders(train_data: torch.utils.data.Dataset,
                       test_data: torch.utils.data.Dataset,
                       batch_size: int,
                       num_workers: int=NUM_WORKERS):

    train_dataloader = torch.utils.data.DataLoader(train_data, batch_size=batch_size,
                                                   shuffle=True, num_workers=num_workers,
                                                   pin_memory=torch.cuda.is_available())
    test_dataloader = torch.utils.data.DataLoader(test_data, batch_size=batch_size,
                                                  shuffle=False, num_workers=num_workers,
                                                  pin_memory=torch.cuda.is_available())

    return train_dataloader, test_dataloader

Overwriting going_modular/data_setup.py


Scripts written, let's move to the next notebook.